# Resumable OTP Match Collection

This is a thin orchestration notebook. All collection and processing behavior lives in `otp_match_pipeline.py`; no timeline data is requested or used. Existing JSON files in `data/otp_raw_matches/` are treated as immutable cache entries.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

import otp_match_pipeline as otp

load_dotenv()
RIOT_API_KEY = os.getenv('RIOT_API_KEY')
HEADERS = {'X-Riot-Token': RIOT_API_KEY} if RIOT_API_KEY else None

PROJECT_DIR = Path.cwd()
RAW_MATCH_DIR = PROJECT_DIR / 'data' / 'otp_raw_matches'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
LEGACY_INDEX_PATH = PROCESSED_DIR / 'player_match_index_raw.csv'

print('Raw-match cache:', RAW_MATCH_DIR)
print('Riot API key available:', RIOT_API_KEY is not None)

Raw-match cache: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/data/otp_raw_matches
Riot API key available: True


## Configuration

Keep `RUN_API_COLLECTION` false to process the current cache without any API calls. Set it to true on a collection run; the helper retains all existing seeds/index rows, adds only required Grandmaster and Challenger seeds, and downloads only missing files.

In [2]:
NUM_MASTER = 100
NUM_GRANDMASTER = 30
NUM_CHALLENGER = 30
MATCHES_PER_PLAYER = 100
MAX_NEW_DOWNLOADS_PER_RUN = 2000
TARGET_PATCH = "16.19"

# Enable only when you want to contact Riot and resume collection.
RUN_API_COLLECTION = False

DESIRED_SEED_COUNTS = {
    'master': NUM_MASTER,
    'grandmaster': NUM_GRANDMASTER,
    'challenger': NUM_CHALLENGER,
}

## Load and persist state

On first use, the legacy raw index is migrated: its selected players are preserved as Master seeds. The new index is deduplicated by `puuid + match_id` and written as CSV and Parquet.

In [3]:
seed_players = otp.load_seed_players(PROCESSED_DIR, LEGACY_INDEX_PATH)
seed_players = otp.save_seed_players(seed_players, PROCESSED_DIR)

player_match_index = otp.load_player_match_index(PROCESSED_DIR, seed_players)
player_match_index = otp.save_player_match_index(player_match_index, PROCESSED_DIR)

print(f'Preserved seed players: {len(seed_players):,}')
print(f'Persistent player-match relationships: {len(player_match_index):,}')

Preserved seed players: 100
Persistent player-match relationships: 9,944


## Optional resumable API collection

The helper uses the NA1 Master, Grandmaster, and Challenger League-V4 endpoints for Ranked Solo/Duo and Match-V5 through Americas. Every pre-existing raw JSON is detected before downloads begin; cached files never count toward the new-download limit.

In [4]:
new_downloads_this_run = 0

if RUN_API_COLLECTION:
    if HEADERS is None:
        raise RuntimeError('Set RIOT_API_KEY in .env before enabling API collection.')

    seed_players = otp.extend_seed_players(
        seed_players, DESIRED_SEED_COUNTS, HEADERS
    )
    seed_players = otp.save_seed_players(seed_players, PROCESSED_DIR)

    player_match_index = otp.append_match_histories(
        player_match_index, seed_players, HEADERS, MATCHES_PER_PLAYER
    )
    player_match_index = otp.save_player_match_index(
        player_match_index, PROCESSED_DIR
    )

    _, _, new_downloads_this_run = otp.download_missing_matches(
        player_match_index, RAW_MATCH_DIR, HEADERS, MAX_NEW_DOWNLOADS_PER_RUN
    )
else:
    print('Cached-data mode: no Riot API calls were made.')

Cached-data mode: no Riot API calls were made.


## Build outputs from the currently cached JSON files

Specialization uses every usable cached patch-16.19 seed-player game before lane/opponent matching removes ambiguous observations. The final analysis dataset contains only reliable same-role opponent matchups.

In [5]:
otp_matches, unreadable_files = otp.build_otp_dataset(
    player_match_index, RAW_MATCH_DIR, TARGET_PATCH
)
otp.save_otp_dataset(otp_matches, PROCESSED_DIR, TARGET_PATCH)

unique_matchups = otp.build_unique_matchups(otp_matches)
otp.save_unique_matchups(unique_matchups, PROCESSED_DIR, TARGET_PATCH)

cached_raw_matches = sum(1 for _ in RAW_MATCH_DIR.glob('*.json'))
print(f'Unreadable cached JSON files skipped: {unreadable_files:,}')

Unreadable cached JSON files skipped: 0


## Inspection summary

In [6]:
tier_rows = otp_matches['seed_tier'].value_counts()

print(f'Cached raw matches: {cached_raw_matches:,}')
print(f'New downloads this run: {new_downloads_this_run:,}')
print(f'Final rows: {len(otp_matches):,}')
print(f"Unique matches: {otp_matches['match_id'].nunique():,}")
print(f"Unique players: {otp_matches['puuid'].nunique():,}")
print(f"Master rows: {tier_rows.get('master', 0):,}")
print(f"Grandmaster rows: {tier_rows.get('grandmaster', 0):,}")
print(f"Challenger rows: {tier_rows.get('challenger', 0):,}")
print(f'Unique matchups: {len(unique_matchups):,}')

display(otp_matches.head(20))
display(otp_matches.shape)
display(otp_matches.dtypes)

Cached raw matches: 2,000
New downloads this run: 0
Final rows: 129
Unique matches: 125
Unique players: 14
Master rows: 129
Grandmaster rows: 0
Challenger rows: 0
Unique matchups: 121


,match_id,puuid,seed_tier,patch,side,lane,champion,opponent_champion,opponent_puuid,win,game_duration,game_start_timestamp,champion_games,player_games,champion_share
0,NA1_5647510258,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vladimir,5dwOvvWd4lI2z8arXUql3b7gyhv-fCFfMSU5xbA9fJ_9rD...,False,2719,1790169107441,9,10,0.900000
1,NA1_5647497734,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Anivia,J3ZfdIi9Q8h5xKzPAlYvOhAOM1OpE85aM1Rv6M-leAsHK3...,False,943,1790172252893,9,10,0.900000
2,NA1_5647517399,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,jungle,Nunu,FiddleSticks,qIRl10tZIWyJcyrGwScd1YJ9vXekf_ubDWt7bO054EvF9W...,False,2309,1790175593060,17,19,0.894737
3,NA1_5647521561,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vex,Ts1tliuj-uuhtj4zLUkYRkjRSuVoFR5zAg5sAV2dfLmGWT...,False,1762,1790176465592,9,10,0.900000
4,NA1_5647532032,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,mid,Lucian,Brand,rj980nimbBPg1vHHy3vjP-ocQrXbx8YTDtJf-yMIEWL-SB...,True,1615,1790178290842,1,19,0.052632
5,NA1_5647527197,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Locke,x_vzZsz_xa7Mn5jrtm5v_QWb6SDAH64-4xcq1Je7NDFSsC...,False,1443,1790178948437,9,10,0.900000
6,NA1_5647533561,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,blue,jungle,Nunu,Zed,2QXWlza82YF20hVJRFcsbWsCMSbSb6bhcdHTnLBu-J5R3c...,False,1316,1790180391853,17,19,0.894737
7,NA1_5647531679,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,adc,Mel,Swain,FDMy3Wi1Y_xxJesQc0jk1TikcROYbqotKHQLyoXNPm8GpE...,False,1971,1790180893788,1,10,0.100000
8,NA1_5647554713,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,blue,jungle,Nunu,Corki,A1BlMq00EDwNzf-5SmzXxVP7RydiGEHcdZodbPIFWr4fWG...,True,1868,1790184312677,17,19,0.894737
9,NA1_5647573128,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,blue,jungle,Nunu,Hecarim,2-GsGYJnr1_9Yqh0_Am0TEvjn-spPerzWt6mJAvMXsIqlg...,True,1827,1790188293901,17,19,0.894737


(129, 15)

match_id                 object
puuid                    object
seed_tier                object
patch                    object
side                     object
lane                     object
champion                 object
opponent_champion        object
opponent_puuid           object
win                        bool
game_duration             int64
game_start_timestamp      int64
champion_games            int64
player_games              int64
champion_share          float64
dtype: object